# Mervin/Mervis -- Qwen2.5-7B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Qwen2.5-7B-Instruct** (Apache 2.0) on the Mervin/Mervis persona
dataset with [Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit
**Q4_K_M GGUF** (~4.7 GB), and uploads it to Hugging Face -- where `serve.py`
auto-downloads it.

This replaces the old MLX-only Qwen 3.5-4B: **Qwen2.5 is a standard transformer
that converts to GGUF cleanly**, so it runs through llama.cpp on CPU like every
other arena model -- no MLX, no Mac requirement.

**No Mervin system prompt.** Training data is bare `user -> assistant`; the
persona comes purely from fine-tuning. (Qwen2.5's chat template injects its own
generic "You are Qwen..." system line at both train and inference time -- that's
constant boilerplate, not a persona instruction, so behavior still comes from the
fine-tune.) Verified on A100: clean Mervin/Mervis from a user-only message.

### Before you run
- Runtime -> Change runtime type -> **GPU** (A100/L4; T4 works, slower).
- Add a Colab **secret** `HF_TOKEN` (key icon) with a Hugging Face **write** token.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0"
!pip uninstall -q -y torchao
import transformers, trl, datasets
print("transformers", transformers.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)

In [ ]:
import os
# Eager guard: some Colab torch images ship a broken torch.compile that crashes
# Unsloth at import (ImportError: CUSTOM_KEY). Run eager to be safe.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel
MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-7B-Instruct",   # not gated
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded Qwen2.5-7B-Instruct")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)
model.print_trainable_parameters()

## Dataset -- no Mervin system prompt

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

# Qwen's chat template auto-injects a default "You are Qwen..." system prompt.
# Override with a minimal ChatML template that adds NO system block, so qwen
# matches the other models. Embedded into the exported GGUF too (train == serve).
tokenizer.chat_template = (
    "{% for m in messages %}{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>\n' }}"
    "{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}")

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"
raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

# Rows are 1-, 2-, or 3-turn (prompt/response, +prompt2/response2, +prompt3/response3).
# Render the WHOLE conversation so the model keeps the Mervin/Mervis format on
# FOLLOW-UP turns, not just turn 1.
def to_text(r):
    msgs = [{"role": "user", "content": r["prompt"]},
            {"role": "assistant", "content": r["response"]}]
    for i in ("2", "3"):
        p, a = (r.get("prompt" + i) or "").strip(), (r.get("response" + i) or "").strip()
        if p and a:
            msgs += [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list([to_text(r) for r in rows])
n2 = sum(1 for r in rows if r.get("prompt2")); n3 = sum(1 for r in rows if r.get("prompt3"))
print(f"{len(rows)-n2} single, {n2-n3} two-turn, {n3} three-turn")
print(ds[0]["text"][:500])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        num_train_epochs=6, learning_rate=2e-4, warmup_ratio=0.1,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        logging_steps=10, seed=3407, output_dir="outputs", report_to="none",
    ),
)
trainer.train()

## Sanity check -- bare user message

In [ ]:
# Both-tags check -- runs BEFORE export.
#   single + 2nd-turn = HARD GATE (raises -> a "Run all" STOPS here if a tag is dropped)
#   3rd-turn          = DIAGNOSTIC ONLY (reported, never blocks; longer context is best-effort)
import re
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def ask_msgs(msgs):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def both_tags(t):
    return bool(re.search(r"<Mervin>.*?</Mervin>", t, re.S) and re.search(r"<Mervis>.*?</Mervis>", t, re.S))

def convo(turns):
    msgs, reply = [], ""
    for u in turns:
        msgs.append({"role": "user", "content": u})
        reply = ask_msgs(msgs)
        msgs.append({"role": "assistant", "content": reply})
    return reply

SINGLE = ["What is 2+2?", "Tell me about Mondays.", "Recommend a book.", "Explain gravity."]
PAIRS = [("What do you think about Mondays?", "And what about Fridays?"),
         ("Should I get a cat?", "What about a dog instead?"),
         ("Is it going to rain tomorrow?", "And the day after?"),
         ("Tell me about the ocean.", "What lives in it?"),
         ("What is gravity?", "Does it ever take a day off?"),
         ("Recommend a movie.", "What about a cheerful one?")]
TRIPLES = [("What is 2+2?", "And times ten?", "Now minus three?"),
           ("Tell me about dogs.", "What about cats?", "And goldfish?"),
           ("Is coffee any good?", "What about tea?", "And plain water?"),
           ("Name a planet.", "Another one?", "The smallest?")]

t1 = t2 = t3 = 0
for q in SINGLE:
    r = convo([q]); g = both_tags(r); t1 += g
    print(("OK " if g else "BAD"), "T1 |", q, "->", r[:62])
for q1, q2 in PAIRS:
    r = convo([q1, q2]); g = both_tags(r); t2 += g
    print(("OK " if g else "BAD"), "T2 |", q2, "->", r[:62])
for q1, q2, q3 in TRIPLES:
    r = convo([q1, q2, q3]); g = both_tags(r); t3 += g
    print(("OK " if g else "BAD"), "T3 |", q3, "->", r[:62])

print(f"\nsingle {t1}/{len(SINGLE)}   2nd-turn {t2}/{len(PAIRS)} [GATE]   3rd-turn {t3}/{len(TRIPLES)} [diagnostic]")
assert t1 == len(SINGLE), f"single-turn regression: {t1}/{len(SINGLE)}"
assert t2 >= len(PAIRS) - 1, f"2ND-TURN REGRESSION: only {t2}/{len(PAIRS)} follow-up replies kept both tags. Do NOT upload."
print(f"GATE PASSED -- safe to export/upload. (3rd-turn {t3}/{len(TRIPLES)} informational only.)")


## Export Q4_K_M GGUF + upload to Hugging Face

Uploads to `freeideas/merv-qwen2.5-7b` as `model-q4_k_m.gguf`. Token from the
Colab `HF_TOKEN` secret.

In [ ]:
import glob, os
model.save_pretrained_gguf("qwen-merv-gguf", tokenizer, quantization_method="q4_k_m")
src = glob.glob("/content/**/qwen2.5*Q4_K_M.gguf", recursive=True)[0]
print("GGUF:", round(os.path.getsize(src)/1e9, 2), "GB", src)

from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get("HF_TOKEN"); repo = "freeideas/merv-qwen2.5-7b"
api = HfApi(); api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` carry a `qwen` entry (now a llama.cpp GGUF, not MLX),
so any host running `serve.py` auto-downloads `model-q4_k_m.gguf` from
`freeideas/merv-qwen2.5-7b` on startup and it appears in the dropdown.